# Comparison

Compares several `run.py` runs (e.g. different algorithms or network sizes). Two views:

1. **Mean ± 95% CI** across seeds (`plot_comparison`).
2. **Best seed per run** — overlays each run's single best-performing seed, to compare
   best-case behaviour separately from seed-to-seed variance.

Each run directory holds a `training_rewards.npz` with `rewards` `[n_seeds, n_iterations]` and a matching `seeds` array.

In [ ]:
import os
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from utils import plot_comparison

Point these at your run directories (label → path):

In [ ]:
run_dirs = {
    "npg_4x4": "../runs/Hopper/NPG/4x4",
    "npg_8x8": "../runs/Hopper/NPG/8x8",
    "npg_16x16": "../runs/Hopper/NPG/16x16",
}
env_id = "Hopper"
save_dir = "../runs/Hopper"

## Mean ± 95% CI across seeds

In [ ]:
# rewards arrays are [1, n_iterations] (single-seed) or [n_seeds, n_iterations] (multiseed);
# plot_comparison handles both.
rewards_dict = {}

for label, run_dir in run_dirs.items():
    npz_path = os.path.join(run_dir, "training_rewards.npz")
    if os.path.exists(npz_path):
        data = np.load(npz_path)
        rewards_dict[label] = data["rewards"]
        print(f"{label}: rewards {data['rewards'].shape}, seeds = {data['seeds']}")
    else:
        print(f"Warning: missing {npz_path} — skipping {label}")

plot_comparison(rewards_dict, save_dir=save_dir, env_id=env_id)

## Best seed per run

For each run, pick the single best-performing seed and overlay those curves. `metric` chooses how "best" is scored:

- `"final"`: mean return over the last `final_window` iterations (robust to end-of-training spikes).
- `"peak"`: single highest return reached.

In [ ]:
metric = "final"        # "final" or "peak"
final_window = 100

plt.figure(figsize=(8, 5))

for label, run_dir in run_dirs.items():
    npz_path = os.path.join(run_dir, "training_rewards.npz")
    if not os.path.exists(npz_path):
        print(f"Warning: missing {npz_path} — skipping {label}")
        continue

    data = np.load(npz_path)
    rewards, seeds = data["rewards"], data["seeds"]

    if metric == "peak":
        scores = rewards.max(axis=1)
    else:
        w = min(final_window, rewards.shape[1])
        scores = rewards[:, -w:].mean(axis=1)

    best = int(np.argmax(scores))
    curve = rewards[best]
    plt.plot(np.arange(curve.shape[0]), curve,
             label=f"{label} — seed {seeds[best]} ({metric}={scores[best]:.0f})")
    print(f"{label}: best seed {seeds[best]} ({metric} score {scores[best]:.1f})")

plt.xlabel("Iteration")
plt.ylabel("Average training return")
plt.title(f"Best-seed comparison — {env_id}")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "best_seed_comparison.png"), dpi=300)
plt.show()